# Mounting my resources from my google drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.chdir("/content/drive/MyDrive/DSN Hackathon 2026")

In [ ]:
!pip install optuna catboost

# Import packages

In [ ]:
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import QuantileTransformer, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from scipy.optimize import minimize

In [ ]:
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

In [ ]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(f"Train: {train.shape}, Test: {test.shape}")
print(train.dtypes)
print(train.isnull().sum())
print(train['total_sales'].describe())

In [ ]:
train['total_sales'].hist(bins=50)
plt.title('Target Distribution')
plt.show()

print(f"Skewness: {train['total_sales'].skew():.3f}")
print(f"Zeros: {(train['total_sales'] == 0).sum()}")

# Cleaning and categorical normalization

In [ ]:
def normalize_category(s):
    s = s.astype(str).str.strip().str.lower()
    s = s.str.replace("&", "and", regex=False)
    s = s.str.replace(r"\s+", " ", regex=True)
    s = s.str.replace(r"[^a-z0-9_ ]", "", regex=True)
    s = s.str.replace(" ", "_")
    return s

def clean_data(df):
    df = df.copy()

    # IDs and product/store codes keep uppercase and hyphens
    for c in ["product_code", "store_code"]:
        df[c] = df[c].astype(str).str.strip().str.upper()

    # Normalize categorical text
    for c in [
        "fat_content",
        "product_category",
        "store_size",
        "store_location_tier",
        "store_format",
    ]:
        df[c] = normalize_category(df[c])

    # Standardize known categories
    df["fat_content"] = df["fat_content"].replace({
        "low_fat": "low_fat",
        "low fat": "low_fat",
        "regular": "regular",
    })

    df["store_size"] = df["store_size"].replace({
        "small": "small",
        "medium": "medium",
        "large": "large",
        "": "unknown",
        "nan": "unknown",
    })

    df["store_location_tier"] = df["store_location_tier"].replace({
        "tier_1": "tier_1",
        "tier_2": "tier_2",
        "tier_3": "tier_3",
    })

    df["store_format"] = df["store_format"].replace({
        "standard_supermarket": "standard_supermarket",
        "flagship_hypermarket": "flagship_hypermarket",
        "superstore": "superstore",
        "corner_shop": "corner_shop",
    })

    return df

train = clean_data(train)
test = clean_data(test)

# Missing value imputation

In [ ]:
def impute_missing(train, test):
    train = train.copy()
    test = test.copy()

    # Missing indicators BEFORE imputation
    train["product_weight_kg_isna"] = train["product_weight_kg"].isna().astype(int)
    test["product_weight_kg_isna"] = test["product_weight_kg"].isna().astype(int)

    train["store_size_isna"] = train["store_size"].isin(["unknown", "nan", ""]).astype(int)
    test["store_size_isna"] = test["store_size"].isin(["unknown", "nan", ""]).astype(int)

    # Combine for consistent imputation
    train["is_train"] = 1
    test["is_train"] = 0
    df = pd.concat([train, test], axis=0, ignore_index=True)

    # Product weight imputation
    df["product_weight_kg"] = df.groupby("product_code")["product_weight_kg"].transform(
        lambda x: x.fillna(x.median())
    )
    df["product_weight_kg"] = df.groupby("product_category")["product_weight_kg"].transform(
        lambda x: x.fillna(x.median())
    )
    df["product_weight_kg"] = df["product_weight_kg"].fillna(df["product_weight_kg"].median())

    # Store size imputation
    def mode_fill(series):
        m = series.mode()
        return m.iat[0] if len(m) else np.nan

    store_mode = df.groupby("store_code")["store_size"].agg(mode_fill)
    df["store_size"] = df["store_size"].replace(["unknown", "nan", ""], np.nan)
    df["store_size"] = df["store_size"].fillna(df["store_code"].map(store_mode))
    df["store_size"] = df["store_size"].fillna("unknown")

    # Split back
    train = df[df["is_train"] == 1].drop(columns=["is_train"]).reset_index(drop=True)
    test = df[df["is_train"] == 0].drop(columns=["is_train"]).reset_index(drop=True)

    return train, test

train, test = impute_missing(train, test)

# Feature engineering

In [ ]:
def build_features(train, test):
    train = train.copy()
    test = test.copy()

    train["is_train"] = 1
    test["is_train"] = 0
    df = pd.concat([train, test], axis=0, ignore_index=True)

    # ---------- Basic interactions ----------
    df["product_store"] = df["product_code"] + "__" + df["store_code"]
    df["category_format"] = df["product_category"] + "__" + df["store_format"]
    df["category_tier"] = df["product_category"] + "__" + df["store_location_tier"]
    df["category_size"] = df["product_category"] + "__" + df["store_size"]
    df["format_tier"] = df["store_format"] + "__" + df["store_location_tier"]
    df["size_tier"] = df["store_size"] + "__" + df["store_location_tier"]
    df["fat_format"] = df["fat_content"] + "__" + df["store_format"]

    # ---------- Frequency encoding ----------
    freq_cols = [
        "product_code",
        "store_code",
        "product_category",
        "store_format",
        "product_store",
        "category_format",
        "category_tier",
        "category_size",
        "format_tier",
        "size_tier",
        "fat_format",
    ]
    for c in freq_cols:
        df[f"{c}_freq"] = df[c].map(df[c].value_counts())

    # ---------- Numeric interactions ----------
    eps = 1e-6
    df["price_per_kg"] = df["product_price"] / (df["product_weight_kg"] + eps)
    df["price_x_shelf"] = df["product_price"] * df["shelf_visibility"]
    df["weight_x_shelf"] = df["product_weight_kg"] * df["shelf_visibility"]
    df["price_x_weight"] = df["product_price"] * df["product_weight_kg"]
    df["price_per_shelf"] = df["product_price"] / (df["shelf_visibility"] + eps)
    df["weight_per_price"] = df["product_weight_kg"] / (df["product_price"] + eps)
    df["shelf_per_weight"] = df["shelf_visibility"] / (df["product_weight_kg"] + eps)
    df["age_x_price"] = df["store_age_years"] * df["product_price"]
    df["age_x_shelf"] = df["store_age_years"] * df["shelf_visibility"]

    # ---------- Ordinal encodings ----------
    size_map = {"small": 0, "medium": 1, "large": 2, "unknown": -1}
    tier_map = {"tier_1": 0, "tier_2": 1, "tier_3": 2}

    df["store_size_ord"] = df["store_size"].map(size_map).fillna(-1)
    df["store_location_tier_ord"] = df["store_location_tier"].map(tier_map).fillna(-1)

    # ---------- Group statistics without target ----------
    group_cols = [
        "product_code",
        "store_code",
        "product_category",
        "store_format",
        "product_store",
        "category_format",
    ]
    num_cols = ["product_price", "product_weight_kg", "shelf_visibility", "store_age_years"]

    for gcol in group_cols:
        for ncol in num_cols:
            grp = df.groupby(gcol)[ncol]
            df[f"{gcol}_{ncol}_mean"] = grp.transform("mean")
            df[f"{gcol}_{ncol}_std"] = grp.transform("std")
            df[f"{gcol}_{ncol}_dev"] = df[ncol] - df[f"{gcol}_{ncol}_mean"]

    # ---------- Rank features ----------
    for gcol in ["product_category", "store_code", "store_format"]:
        for ncol in ["product_price", "product_weight_kg", "shelf_visibility"]:
            df[f"{ncol}_rank_in_{gcol}"] = df.groupby(gcol)[ncol].rank(pct=True)

    # ---------- Binned features ----------
    df["price_bin"] = pd.qcut(df["product_price"], q=10, duplicates="drop", labels=False)
    df["weight_bin"] = pd.qcut(df["product_weight_kg"], q=10, duplicates="drop", labels=False)
    df["shelf_bin"] = pd.qcut(df["shelf_visibility"], q=10, duplicates="drop", labels=False)
    df["age_bin"] = pd.cut(
        df["store_age_years"],
        bins=[0, 10, 20, 30, 40, 50, 100],
        labels=False,
    )

    # Split back
    train_fe = df[df["is_train"] == 1].drop(columns=["is_train"]).reset_index(drop=True)
    test_fe = df[df["is_train"] == 0].drop(columns=["is_train"]).reset_index(drop=True)

    return train_fe, test_fe

train_fe, test_fe = build_features(train, test)

# Log target for training and target encoding
train_fe["total_sales_log"] = np.log1p(train_fe["total_sales"])

# Leakage-safe out-of-fold target encoding

In [ ]:
def add_target_encoding(
    tr,
    va,
    te,
    te_cols,
    smooth_dict,
    target_col="total_sales_log",
    n_inner=5,
    seed=SEED,
):
    tr = tr.copy()
    va = va.copy()
    te = te.copy()

    global_mean = tr[target_col].mean()

    for col in te_cols:
        smooth = smooth_dict.get(col, 20)
        tr[f"te_{col}"] = np.nan

        # Inner OOF for training rows
        kf = KFold(n_splits=n_inner, shuffle=True, random_state=seed)
        for inner_tr_idx, inner_va_idx in kf.split(tr):
            inner_tr = tr.iloc[inner_tr_idx]
            inner_va = tr.iloc[inner_va_idx]

            agg = inner_tr.groupby(col)[target_col].agg(["mean", "count"])
            smap = (
                agg["mean"] * agg["count"] + global_mean * smooth
            ) / (agg["count"] + smooth)

            tr.loc[tr.index[inner_va_idx], f"te_{col}"] = (
                inner_va[col].map(smap).fillna(global_mean)
            )

        # Full mapping from this fold's training data for validation/test
        agg = tr.groupby(col)[target_col].agg(["mean", "count"])
        smap = (
            agg["mean"] * agg["count"] + global_mean * smooth
        ) / (agg["count"] + smooth)

        va[f"te_{col}"] = va[col].map(smap).fillna(global_mean)
        te[f"te_{col}"] = te[col].map(smap).fillna(global_mean)

    return tr, va, te

In [ ]:
te_cols = [
    "product_code",
    "store_code",
    "product_store",
    "product_category",
    "store_format",
    "category_format",
    "category_tier",
    "category_size",
    "format_tier",
    "size_tier",
    "fat_format",
]

smooth_dict = {
    "product_code": 30,
    "store_code": 30,
    "product_store": 100,
    "product_category": 20,
    "store_format": 20,
    "category_format": 50,
    "category_tier": 50,
    "category_size": 50,
    "format_tier": 50,
    "size_tier": 50,
    "fat_format": 30,
}

#Preparing XGBoost matrix: one-hot low cardinality, target-encode high cardinality

In [ ]:
# ------------------------------------------------------------------
# SECTION 5 — Prepare XGBoost-ready matrices (idempotent)
# ------------------------------------------------------------------
# Rebuild raw feature frames from Section 3 output so re-running this
# cell does NOT accumulate one-hot columns from previous runs.
train_raw = build_features(train, test)   # returns (train_fe_raw, test_fe_raw)
train_raw, test_raw = train_raw

# Ensure log target exists on the raw training frame
train_raw["total_sales_log"] = np.log1p(train_raw["total_sales"])

# CatBoost copies (raw categorical strings kept intact)
train_cb = train_raw.copy()
test_cb = test_raw.copy()

# ---- low/high cardinality lists ----
low_card = [
    "fat_content",
    "store_size",
    "store_location_tier",
]

high_card = [
    "product_code",
    "store_code",
    "product_store",
    "product_category",
    "store_format",
    "category_format",
    "category_tier",
    "category_size",
    "format_tier",
    "size_tier",
    "fat_format",
]

# ---- sanity checks ----
missing_low = [c for c in low_card if c not in train_raw.columns]
if missing_low:
    raise ValueError(
        f"low_card columns missing from raw train frame: {missing_low}. "
        f"You may have run a later cell that already one-hot encoded them. "
        f"Re-run Section 3 (build_features) first."
    )
assert not (set(low_card) & set(te_cols)), set(low_card) & set(te_cols)
assert not (set(low_card) & set(high_card)), set(low_card) & set(high_card)

# ---- one-hot on a fresh combined frame ----
combined = pd.concat([train_raw, test_raw], axis=0, ignore_index=True)
combined = pd.get_dummies(combined, columns=low_card, drop_first=False)

train_fe = combined.iloc[: len(train_raw)].copy()
test_fe = combined.iloc[len(train_raw):].copy()

# Columns to drop from XGBoost input AFTER target encoding is added
drop_for_xgb = ["id", "total_sales", "total_sales_log"] + high_card

# XGBoost CV with target encoding inside each fold

In [ ]:
from sklearn.metrics import mean_squared_error

def xgb_cv(params, train_df, test_df, n_splits=10):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    oof_log = np.zeros(len(train_df))
    test_preds = []

    for fold, (tr_idx, va_idx) in enumerate(kf.split(train_df)):
        tr = train_df.iloc[tr_idx].copy()
        va = train_df.iloc[va_idx].copy()
        te = test_df.copy()

        tr, va, te = add_target_encoding(
            tr, va, te, te_cols, smooth_dict, target_col="total_sales_log"
        )

        features = [c for c in tr.columns if c not in drop_for_xgb]
        X_tr = tr[features]
        y_tr = tr["total_sales_log"]
        X_va = va[features]
        y_va = va["total_sales_log"]

        model = xgb.XGBRegressor(
            **params,
            objective="reg:squarederror",
            eval_metric="rmse",
            tree_method="hist",
            random_state=SEED,
            n_jobs=-1,
            early_stopping_rounds=100,
            verbosity=0,
        )

        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_va, y_va)],
            verbose=False,
        )

        best_it = getattr(model, "best_iteration", None)
        if best_it is not None:
            oof_log[va_idx] = model.predict(X_va, iteration_range=(0, best_it + 1))
            test_preds.append(
                model.predict(te[features], iteration_range=(0, best_it + 1))
            )
        else:
            oof_log[va_idx] = model.predict(X_va)
            test_preds.append(model.predict(te[features]))

    oof_orig = np.expm1(oof_log)
    rmse_val = float(np.sqrt(mean_squared_error(train_df["total_sales"].values, oof_orig)))

    return oof_orig, np.expm1(np.mean(test_preds, axis=0)), rmse_val

# Optuna tuning for XGBoost

In [ ]:
def xgb_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 500, 2500),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.08, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 30),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 5.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 20.0),
    }

    _, _, rmse = xgb_cv(params, train_fe, test_fe, n_splits=10)
    return rmse

xgb_study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
)
xgb_study.optimize(xgb_objective, n_trials=40)

print("Best XGB RMSE:", xgb_study.best_value)
print("Best XGB params:", xgb_study.best_params)

xgb_oof, xgb_test, xgb_rmse = xgb_cv(
    xgb_study.best_params, train_fe, test_fe, n_splits=10
)

# CatBoost CV with native categorical handling

In [ ]:
cat_features = [
    "product_code",
    "store_code",
    "product_category",
    "fat_content",
    "store_size",
    "store_location_tier",
    "store_format",
    "product_store",
    "category_format",
    "category_tier",
    "category_size",
    "format_tier",
    "size_tier",
    "fat_format",
]

def cat_cv(params, train_df, test_df, n_splits=10):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    oof_log = np.zeros(len(train_df))
    test_preds = []

    drop_cols = ["id", "total_sales", "total_sales_log"]

    for fold, (tr_idx, va_idx) in enumerate(kf.split(train_df)):
        tr = train_df.iloc[tr_idx].copy()
        va = train_df.iloc[va_idx].copy()
        te = test_df.copy()

        features = [c for c in tr.columns if c not in drop_cols]
        X_tr = tr[features]
        y_tr = tr["total_sales_log"]
        X_va = va[features]
        y_va = va["total_sales_log"]

        model = CatBoostRegressor(
            **params,
            loss_function="RMSE",
            random_seed=SEED,
            verbose=False,
        )

        model.fit(
            X_tr,
            y_tr,
            cat_features=cat_features,
            eval_set=(X_va, y_va),
            early_stopping_rounds=100,
            use_best_model=True,
        )

        oof_log[va_idx] = model.predict(X_va)
        test_preds.append(model.predict(te[features]))

    oof_orig = np.expm1(oof_log)
    rmse_val = float(np.sqrt(mean_squared_error(train_df["total_sales"].values, oof_orig)))

    return oof_orig, np.expm1(np.mean(test_preds, axis=0)), rmse_val

# Optuna tuning for CatBoost

In [ ]:
def cat_objective(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 500, 3000),
        "depth": trial.suggest_int("depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.08, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 20.0),
        "random_strength": trial.suggest_float("random_strength", 0.0, 5.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 2.0),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "grow_policy": trial.suggest_categorical(
            "grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"]
        ),
    }

    _, _, rmse = cat_cv(params, train_cb, test_cb, n_splits=10)
    return rmse

cat_study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
)
cat_study.optimize(cat_objective, n_trials=30)

print("Best CatBoost RMSE:", cat_study.best_value)
print("Best CatBoost params:", cat_study.best_params)

cat_oof, cat_test, cat_rmse = cat_cv(
    cat_study.best_params, train_cb, test_cb, n_splits=10
)

In [ ]:
# Cell 14b — LightGBM CV (new)
import lightgbm as lgb

def lgb_cv(params, train_df, test_df, n_splits=10):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    oof_log = np.zeros(len(train_df))
    test_preds = []

    for fold, (tr_idx, va_idx) in enumerate(kf.split(train_df)):
        tr = train_df.iloc[tr_idx].copy()
        va = train_df.iloc[va_idx].copy()
        te = test_df.copy()

        tr, va, te = add_target_encoding(tr, va, te, te_cols, smooth_dict)

        features = [c for c in tr.columns if c not in drop_for_xgb]

        model = lgb.LGBMRegressor(
            **params,
            objective="regression",
            metric="rmse",
            random_state=SEED,
            n_jobs=-1,
            verbosity=-1,
        )
        model.fit(
            tr[features], tr["total_sales_log"],
            eval_set=[(va[features], va["total_sales_log"])],
            callbacks=[lgb.early_stopping(100, verbose=False)],
        )

        oof_log[va_idx] = model.predict(
            va[features], num_iteration=model.best_iteration_
        )
        test_preds.append(
            model.predict(te[features], num_iteration=model.best_iteration_)
        )

    oof_orig = np.expm1(oof_log)
    rmse_val = float(np.sqrt(mean_squared_error(
        train_df["total_sales"].values, oof_orig
    )))
    return oof_orig, np.expm1(np.mean(test_preds, axis=0)), rmse_val


# Cell 14c — run LGB with light hyperparameters (no Optuna needed)
lgb_params = {
    "n_estimators": 2000,
    "learning_rate": 0.02,
    "num_leaves": 63,
    "min_child_samples": 20,
    "subsample": 0.85,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
}

lgb_oof, lgb_test, lgb_rmse = lgb_cv(lgb_params, train_fe, test_fe, n_splits=10)
print("LGB OOF RMSE:", lgb_rmse)

# Ensemble XGBoost + CatBoost with OOF-optimized weights

In [ ]:
y_true = train_fe["total_sales"].values

def ensemble_loss(weights):
    w = weights / weights.sum()
    pred = w[0] * xgb_oof + w[1] * cat_oof
    return float(np.sqrt(mean_squared_error(y_true, pred)))

res = minimize(
    ensemble_loss,
    x0=[0.5, 0.5],
    bounds=[(0, 1), (0, 1)],
    method="Nelder-Mead",
)

best_w = res.x / res.x.sum()
print("Best ensemble weights:", best_w)

ensemble_oof = best_w[0] * xgb_oof + best_w[1] * cat_oof
# ---- Fix 1: bias correction for log back-transform ----
correction = y_true.mean() / ensemble_oof.mean()
print("Correction factor:", correction)
ensemble_oof_corrected = ensemble_oof * correction
print("Corrected ensemble OOF RMSE:",
      float(np.sqrt(mean_squared_error(y_true, ensemble_oof_corrected))))

ensemble_test = best_w[0] * xgb_test + best_w[1] * cat_test

# Clip to sensible range
ensemble_test = ensemble_test * correction   # <-- add this line
ensemble_test = np.clip(ensemble_test, 0, train_fe["total_sales"].max() * 1.05)

In [ ]:
y_true = train_fe["total_sales"].values

def ensemble_loss(w):
    w = w / w.sum()
    pred = w[0]*xgb_oof + w[1]*cat_oof + w[2]*lgb_oof
    return float(np.sqrt(mean_squared_error(y_true, pred)))

res = minimize(ensemble_loss, x0=[0.34, 0.33, 0.33],
               bounds=[(0, 1)]*3, method="Nelder-Mead")
best_w = res.x / res.x.sum()

ensemble_oof_1 = best_w[0]*xgb_oof + best_w[1]*cat_oof + best_w[2]*lgb_oof
# ---- Fix 1: bias correction for log back-transform ----
correction = y_true.mean() / ensemble_oof.mean()
print("Correction factor:", correction)
ensemble_oof_corrected = ensemble_oof * correction
print("Corrected ensemble OOF RMSE:",
      float(np.sqrt(mean_squared_error(y_true, ensemble_oof_corrected))))

ensemble_test_1 = best_w[0]*xgb_test + best_w[1]*cat_test + best_w[2]*lgb_test

# Clip to sensible range
ensemble_test_1 = ensemble_test_1 * correction   # <-- add this line
ensemble_test_1 = np.clip(ensemble_test, 0, train_fe["total_sales"].max() * 1.05)

# Submission

In [ ]:
submission = pd.DataFrame({
    "id": test["id"],
    "total_sales": ensemble_test_1,
})

submission.to_csv("submission_final.csv", index=False)
print(submission.head())